# API RESPONSE DESIGN
This jupyter will be a key part to design the desired output of each endpoint. First step is to exactly know what information will be needed by front, and structure it accordingly.
This top-down approach will avoid having un-manageable JSON responses. At the same time, this will allow to do a front-end design using mock data.

### Load dependencies

In [1]:
import pandas as pd
import re
import json

### Load data

In [2]:
from src.api.paths import DATA_SOURCE

sport = pd.read_excel(DATA_SOURCE, sheet_name="Escalada")
boulder = pd.read_excel(DATA_SOURCE, sheet_name="Bloque")
trad = pd.read_excel(DATA_SOURCE, sheet_name="Clásica")
multi = pd.read_excel(DATA_SOURCE, sheet_name="Largos")
climbing_data = {
    "sport": sport,
    "boulder": boulder,
    "trad": trad,
    "multi": multi,
}

C:\Users\Alonso\AppData\Local\Programs\Python\Python310\lib\site-packages\openpyxl\worksheet\_reader.py:312: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


# Preprocessing

## Columns translation

In [3]:
equivalences = {
    "Lugar": "zone",
    "Sector": "crag",
    "Vía": "route",
    "Fecha": "date",
    "Grado": "grade",
    "Percepción": "perception",
    "Pegue": "tries",
    "Calidad": "quality",
    "Comentario": "comment",
}
sport = sport.rename(equivalences, axis=1)
boulder = boulder.rename(equivalences, axis=1)

In [4]:
equivalences = {
    "Lugar": "zone",
    "Sector": "crag",
    "Vía": "route",
    "Fecha": "date",
    "Grado": "grade",
    "Largos": "pitches",
    "Longitud": "length",
    "Resultado": "result",
    "Tiempo": "duration",
    "Comentario": "comment",
}
trad = trad.rename(equivalences, axis=1)
multi = multi.rename(equivalences, axis=1)

## Tries translation

In [5]:
equivalences = {
    "Top Rope": "Top Rope",
    "Vista": "OS",
    "Flash": "Flash",
    "Segundo": "Second",
    "Tercero": "Third",
    "No encadeno": "Not sent",
    "Repetición": "Repetition",
    "Proyecto": "Project",
}

sport = sport.replace({"tries": equivalences})
boulder = boulder.replace({"tries": equivalences})

In [6]:
sport

,zone,crag,route,date,grade,perception,tries,quality,comment
0,Majadilla verde,Izquierda,-,2020-08-19,NaN,NaN,NaN,NaN,Vías cortas y reunión accesible andando
1,Majadilla verde,Izquierda,-,2020-09-05,NaN,NaN,NaN,NaN,Vías cortas y reunión accesible andando
2,Patones,Gajo free,Bayerina Electrónica,2020-09-12,6a,6a,Top Rope,NaN,La monta Jerome
3,Patones,Gajo free,Topez lopez,2020-09-12,6a,6a,Second,NaN,Tras probarla en top rope le doy de primero
4,Patones,Gajo free,El gato montés,2020-09-12,6a+,6a+,Top Rope,NaN,La monta Jerome
...,...,...,...,...,...,...,...,...,...
803,Cueva del Arenal,NaN,Rodillar,2024-05-18,7c,7c,Not sent,5,Trabajando. Paso chapa 4 que no me acordaba y ...
804,Cueva del Arenal,NaN,La petagleta,2024-05-25,6b+,6b+,Repetition,NaN,Para calentar
805,Cueva del Arenal,NaN,Rodillar,2024-05-25,7c,7c,Not sent,5,"Trabajando. Me siento flojo, mejoro chapa 4."
806,San Martín de Valdeiglesias,Vacaciones en el mar,Volando entre bloques,2024-06-16,V+,V+,Repetition,NaN,Calentando


## Extract aid grade for trad/multi-pitch

In [7]:
trad["aid"] = trad["result"].str.extract(r'\/(?P<aid>A\d)$').fillna("free")
trad["result"] = trad["result"].fillna("abandoned")

multi["aid"] = multi["result"].str.extract(r'\/(?P<aid>A\d)$').fillna("free")
multi["result"] = multi["result"].fillna("abandoned")

trad[["aid", "result"]]

,aid,result
0,free,Dos largos
1,free,V
2,free,IV+
3,free,V
4,free,IV+
5,free,6a
6,free,IV+
7,free,6a+
8,free,6a
9,free,V


In [8]:
re_grade = r'^(?:\d[ABCabc])|(?:[IV]{1,2})\+?$'
re_aid = r'A\d$'

def compute_result(row) -> str:
    if re.match(re_aid, row["aid"]):
        return f"{row['grade']}/{row['aid']}"
    elif re.match(re_grade, row["result"]):
        return row["grade"]

    return "abandoned"

trad["result"] = trad.apply(compute_result, axis=1)
multi["result"] = multi.apply(compute_result, axis=1)

trad["result"]

0     abandoned
1             V
2           IV+
3             V
4           IV+
5            6a
6           IV+
7           6a+
8            6a
9             V
10           V+
11           V+
12          IV+
13           V+
14           6a
15            V
16           6b
17    abandoned
18            V
19           V+
20          IV+
21           6b
22          IV+
23    abandoned
24           6a
25           V+
26          6a+
27           6a
28        7a/A1
29           V+
30           6a
Name: result, dtype: object

In [9]:
multi["result"]

0            IV
1            IV
2         6a/A0
3     abandoned
4     abandoned
5     abandoned
6            V+
7             V
8            6a
9             V
10    abandoned
11    abandoned
12    abandoned
13          IV+
14           V+
15            V
16            V
17    abandoned
18    abandoned
19           6a
20            V
21          IV+
22            V
23           V+
24           V+
25        V+/A0
26          6a+
27          6a+
28          6a+
29       6a+/A0
Name: result, dtype: object

# Mock responses

## History
Just all the climbed routes
```
{
    sport: [
        {
            crag: "Pedriza",
            sector: "Quebrantaherraduras superior",
            name: "diedro azul",
            date: "2022-04-09",
            tries: "third",
            comment: "my first 7a",
            quality: 5,
            grade: "7a",
            perception: "7a",
        },
        ...
    ],
    boulder: [...], // same as sport
    trad: [
        {
            crag: "riglos",
            sector: "frechín",
            route: "currucuclillo",
            date: "2024-06-03",
            grade: "6a+",
            aid: "A0",  // can be "free"
            pitches: 9,
            length: 230, // meters
            duration: "4:25:00",
            comment: "not a good day"
        },
        ...
    ],
    multi: [...],  // same as trad
}
```

In [10]:
history = {
    "sport": sport.fillna("").to_dict(orient="records"),
    "boulder": boulder.fillna("").to_dict(orient="records"),
    "trad": trad.fillna("").to_dict(orient="records"),
    "multi": multi.fillna("").to_dict(orient="records"),
}

def custom_serializer(obj):
    """Custom JSON serializer for objects not serializable by default json code"""
    if isinstance(obj, pd.Timestamp):
        return obj.strftime('%Y-%m-%d %H:%M:%S')
    else:  # to avoid weird problems with non serializable objects
        return None

with open("mock_history.json", "w", encoding="utf-8") as f:
    f.write(json.dumps(history, default=custom_serializer))

## Calendar
This must contain data for
- sport / trad / multi-pitch
- days -> list in which there is any activity
- routes -> each grade and number of routes in that grade
- statistics ->
```
{
    sport: {
        days: {
            january: [0, 1, 1, 0, 1, 1, 0, 0, 0, 0],
            ...
        }
        routes: {6a+: 5, 6b: 9, 6c: 3, 7a: 1},
        labels: {
            max_grade: [],  // one per month
            max_os_grade: [],  // one per month
            total_days: [],  // one per month
            total_routes: [],  // one per month
        }
    },
    trad: [],
    multi: [],
}
```

### Days

In [11]:
class CalendarData:
    def __init__(self, df: pd.DataFrame):
        self.dates: pd.DataFrame = self.build_dates_df(df)
        self.grades: pd.DataFrame = self.build_grades_df(df)

    def build_dates_df(self, df: pd.DataFrame) -> pd.DataFrame:
        dates = pd.DataFrame({"date": df.date.unique()}).dropna()
        dates["year"] = dates["date"].dt.year
        dates["month"] = dates["date"].dt.month
        dates["day"] = dates["date"].dt.day
        return dates

    def build_grades_df(self, df: pd.DataFrame) -> pd.DataFrame:
        # grades = df[["grade", "date", "tries"]].dropna()
        grades = df.dropna()
        grades["year"] = grades["date"].dt.year
        grades["month"] = grades["date"].dt.month
        grades["day"] = grades["date"].dt.day
        return grades

calendar_data = CalendarData(sport)

C:\Users\Alonso\AppData\Local\Temp\ipykernel_14200\1183014363.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  grades["year"] = grades["date"].dt.year
C:\Users\Alonso\AppData\Local\Temp\ipykernel_14200\1183014363.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  grades["month"] = grades["date"].dt.month
C:\Users\Alonso\AppData\Local\Temp\ipykernel_14200\1183014363.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer]

In [12]:
dates = calendar_data.dates

In [13]:
grades = calendar_data.grades

In [14]:
from calendar import monthrange

def build_empty_calendar(dates_df: pd.DataFrame):
    empty_calendar = {}

    for yyyy in dates_df.year.unique():
        days_in_month = lambda y, m: monthrange(y, m)[1]

        empty_calendar[int(yyyy)] = {
            mm: [0] * days_in_month(yyyy, mm) for mm in range(1, 13)
        }

    return empty_calendar


In [15]:
def build_calendar(dates_df: pd.DataFrame):
    calendar = build_empty_calendar(dates_df)

    for _, row in dates_df.iterrows():
        yyyy, mm, dd = row.year, row.month, row.day
        calendar[int(yyyy)][int(mm)][int(dd)-1] = 1

    return calendar


In [16]:
calendar = build_calendar(dates)
for year, month in zip(dates["year"], dates["month"]):
    print(year, month, calendar[year][month])

2020 8 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2020 9 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
2020 9 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
2020 9 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
2020 9 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
2020 10 [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
2020 10 [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
2020 10 [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
2020 11 [1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]
2020 11 [1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]
202

#### Grades

**todo:** filter to only apply on sent routes

In [17]:
grades

,zone,crag,route,date,grade,perception,tries,quality,comment,year,month,day
13,Patones,Zona de Arriba,5,2020-09-26,IV-,IV-,OS,3,A vaya sitio fácil nos lleva Eloy,2020,9,26
38,San Martín de Valdeiglesias,Barrio Sésamo,Arrea,2020-11-02,V+,V+,Top Rope,3,"Primer pegue me paro a mirar. Luego, me cago a...",2020,11,2
53,Pedriza,Gusarapo,El techo del Tito,2020-12-09,6a,6a,OS,3,Pegue con la nieve cayendo. Mini-escaqueo final?,2020,12,9
74,La Panocha,Paisaje Lunar,Fisura de la Sinfonía,2020-12-28,V,V,OS,2,Vía incómoda,2020,12,28
85,Pedriza,Gusarapo,Placa de LOU,2021-04-04,6a,6a,OS,5,"No sale en la guía, la gente dice que 6a",2021,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...
772,Torrelodones,Desde aquí se ve la fría,2,2024-04-07,IV+,IV+,OS,2,Salir con la segunda que te matas,2024,4,7
773,Torrelodones,Desde aquí se ve la fría,3,2024-04-07,V+,IV+,OS,3,Salir con la segunda que te matas,2024,4,7
774,Torrelodones,Desde aquí se ve la fría,4,2024-04-07,V,V+,OS,4,Bastante buena para lo corta que es.,2024,4,7
776,Torrelodones,Desde aquí se ve la fría,6,2024-04-07,6a,6b,Top Rope,2,Buen arreón hay que meter al inicio sin tocar ...,2024,4,7


In [18]:
def build_grades_hist(grades_df: pd.DataFrame):
    grades_hist = {}

    # Compute hist
    for yyyy in grades_df.year.unique():
        grades_hist[int(yyyy)] = {}

        for mm in grades_df.month.unique():
            grades_month = grades_df[
                (grades_df.month == mm) &
                (grades_df.year == yyyy)
            ]
            grades_hist[int(yyyy)][int(mm)] = grades_month["grade"].value_counts().to_dict()
    return grades_hist


# Print hist
_grades_hist = build_grades_hist(grades)
for year in grades.year.unique():
    for month in grades.month.unique():
        print(year, month, _grades_hist[year][month])

2020 9 {'IV-': 1}
2020 11 {'V+': 1}
2020 12 {'6a': 1, 'V': 1}
2020 4 {}
2020 5 {}
2020 7 {}
2020 8 {}
2020 10 {}
2020 1 {}
2020 2 {}
2020 3 {}
2020 6 {}
2021 9 {'V': 1, '6b': 1, '6a': 1, '6a+': 1}
2021 11 {'IV': 2, 'V+': 1, '6b+': 1}
2021 12 {'6b': 3, '6a+': 3, 'IV': 1, '6b+': 1, 'IV-': 1, '6a': 1, 'V': 1}
2021 4 {'6a': 1, '6b': 1}
2021 5 {'IV+': 1, '6b': 1, '6a+': 1}
2021 7 {'6a': 2, 'V+': 1}
2021 8 {'6a': 2, 'V+': 1}
2021 10 {'6b': 2, '6a+': 2, 'V+': 1, '6a': 1, 'V': 1}
2021 1 {}
2021 2 {}
2021 3 {}
2021 6 {}
2022 9 {'7a': 2, 'III+': 1, 'V+': 1, '6c': 1, '6a+': 1, 'V': 1, '6b': 1, '6a': 1}
2022 11 {'V': 4, '6b+': 2, '6b': 2, 'V+': 2, 'V-': 2, 'IV+': 1, '6a+': 1}
2022 12 {'6a': 5, 'V+': 2, '6c+': 2, '6b': 2, '6b+': 2, '7a': 1, '6c': 1, '7b': 1}
2022 4 {'6b': 3, '6c': 3, '7a': 2, '6a+': 1, '6a': 1}
2022 5 {'6b+': 3, '6b': 3, '7a': 3, 'V+': 2, '6a': 2, '6a+': 2, '6c+': 2, '6c': 2, 'V': 1}
2022 7 {'6a+': 6, '6b': 3, '6b+': 2, '6a': 2, '7a': 1, '6c+': 1, '6c/+': 1, '6c': 1}
2022 8 {'6a': 

#### Routes

In [19]:
def build_routes_hist(grades_df: pd.DataFrame):
    routes_hist = {}

    # Compute hist
    for yyyy in grades_df.year.unique():
        routes_hist[int(yyyy)] = {}

        for mm in grades_df.month.unique():
            routes_month = grades_df[
                (grades_df.month == mm) &
                (grades_df.year == yyyy)
            ]
            routes_hist[int(yyyy)][int(mm)] = len(routes_month)
    return routes_hist


# Print hist
_routes_hist = build_routes_hist(grades)
for year in grades.year.unique():
    for month in grades.month.unique():
        print(year, month, _routes_hist[year][month])

2020 9 1
2020 11 1
2020 12 2
2020 4 0
2020 5 0
2020 7 0
2020 8 0
2020 10 0
2020 1 0
2020 2 0
2020 3 0
2020 6 0
2021 9 4
2021 11 4
2021 12 11
2021 4 2
2021 5 3
2021 7 3
2021 8 3
2021 10 7
2021 1 0
2021 2 0
2021 3 0
2021 6 0
2022 9 9
2022 11 14
2022 12 16
2022 4 10
2022 5 20
2022 7 17
2022 8 9
2022 10 4
2022 1 12
2022 2 11
2022 3 6
2022 6 13
2023 9 5
2023 11 2
2023 12 17
2023 4 5
2023 5 7
2023 7 0
2023 8 5
2023 10 0
2023 1 5
2023 2 3
2023 3 3
2023 6 0
2024 9 0
2024 11 0
2024 12 0
2024 4 4
2024 5 1
2024 7 0
2024 8 0
2024 10 0
2024 1 5
2024 2 3
2024 3 0
2024 6 0


#### Load trad data for easy development

In [20]:
trad_data = CalendarData(trad).grades
trad_data

C:\Users\Alonso\AppData\Local\Temp\ipykernel_14200\1183014363.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  grades["year"] = grades["date"].dt.year
C:\Users\Alonso\AppData\Local\Temp\ipykernel_14200\1183014363.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  grades["month"] = grades["date"].dt.month
C:\Users\Alonso\AppData\Local\Temp\ipykernel_14200\1183014363.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer]

,zone,crag,route,date,grade,pitches,length,result,duration,comment,aid,year,month,day
4,Pedriza,Tortuga,ASA,2023-04-23,IV+,5,170,IV+,02:30:00,Solo metí 4 friends y alguna reu en puente de ...,free,2023,4,23
8,Cabrera,Pico de la Miel,Piloto,2023-05-13,6a,5,150,6a,04:00:00,Tiempo aproximado. 5:30:00 desde salida del pa...,free,2023,5,13
11,Pedriza,Pared de Santillana,Sur clásica,2023-05-19,V+,2,115,V+,03:00:00,La hacemos en dos largos de 60 y 55m más o men...,free,2023,5,19
12,Pedriza,Tortuga,ASA,2023-06-13,IV+,5,170,IV+,01:45:00,Repetición con Alejandro. Esta vez los largos ...,free,2023,6,13
14,Cabrera,Pico de la Miel,Piloto,2023-06-18,6a,5,150,6a,02:45:00,• La hago con Dani cuando para la lluvia y vem...,free,2023,6,18
15,Pedriza,Placas del Halcón,Premio en la cumbre,2023-07-08,V,3,90,V,02:15:00,Muy facilita. En el primer largo pongo algún f...,free,2023,7,8
19,Ifach,Sur,Valencianos,2023-10-15,V+,8,230,V+,03:05:00,L2 hay q ir muy a la izquierda y luego muy a l...,free,2023,10,15
20,Ifach,Norte,Pany,2023-10-15,IV+,7,240,IV+,02:50:00,Alejes asumibles pero mejor llevar algún frien...,free,2023,10,15
21,Ifach,Sur,Diedro UBSA + Miserables,2023-10-16,6b,9,250,6b,06:00:00,Tras el rapel intermedio subimos por el diedro...,free,2023,10,16
22,Pedriza,Tortuga,ASA,2023-12-10,IV+,5,170,IV+,03:30:00,"Con Estefanía, tiempo incuyendo rápel. No lo m...",free,2023,12,10


#### Pitches

In [21]:
# todo: I do not find why there are repeated months in the empty calendar
_empty_calendar = build_empty_calendar(trad_data)
for year, month in zip(trad_data["year"], trad_data["month"]):
    print(year, month, _empty_calendar[year][month])

2023 4 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 5 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 5 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 6 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 6 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 7 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 10 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 10 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 10 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 12 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [22]:
def build_pitches_calendar(data: pd.DataFrame):
    if "pitches" not in data:
        return None
    calendar = build_empty_calendar(data)

    for _, row in data[["year", "month", "day"]].iterrows():
        yyyy, mm, dd = row.year, row.month, row.day
        calendar[int(yyyy)][int(mm)][int(dd)-1] = sum(
            data[
                (data.year == yyyy) &
                (data.month == mm) &
                (data.day == dd)
            ].pitches
        )

    return calendar

calendar = build_pitches_calendar(trad_data)
for year, month in zip(trad_data["year"], trad_data["month"]):
    print(year, month, calendar[year][month])


2023 4 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0]
2023 5 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 5 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 6 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 6 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 7 [0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 10 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 15, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 10 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 15, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 10 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 15, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 12 [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

#### Altitude

In [23]:
def build_altitude_calendar(data: pd.DataFrame):
    if "length" not in data:
        return None

    calendar = build_empty_calendar(data)

    for _, row in data.iterrows():
        yyyy, mm, dd = row.year, row.month, row.day
        calendar[int(yyyy)][int(mm)][int(dd)-1] = sum(
            data[
                (data.year == yyyy) &
                (data.month == mm) &
                (data.day == dd)
            ].length
        )

    return calendar

calendar = build_altitude_calendar(trad_data)
for year, month in zip(trad_data["year"], trad_data["month"]):
    print(year, month, calendar[year][month])


2023 4 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 170, 0, 0, 0, 0, 0, 0, 0]
2023 5 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 150, 0, 0, 0, 0, 0, 115, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 5 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 150, 0, 0, 0, 0, 0, 115, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 6 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 170, 0, 0, 0, 0, 150, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 6 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 170, 0, 0, 0, 0, 150, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 7 [0, 0, 0, 0, 0, 0, 0, 90, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 10 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 470, 250, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 10 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 470, 250, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 10 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 470, 250, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
2023 12 [0, 0, 0, 0, 0, 0, 0, 0, 0, 170, 0, 0, 0, 0, 0, 0, 0, 0, 0

### Labels

#### Max grade

In [24]:
def build_max_grade(grades_df: pd.DataFrame):
    max_grades = {}

    for yyyy in grades_df.year.unique():
        max_grades[int(yyyy)] = {}

        for mm in grades_df.month.unique():
            grades_month = grades_df[
                (grades_df.month == mm) &
                (grades_df.year == yyyy)
            ].grade

            try:
                max_grade_in_month = max(grades_month, key=lambda x: (x[0].isdigit(), x))
            except ValueError:
                max_grade_in_month = None
            max_grades[int(yyyy)][int(mm)] = max_grade_in_month
    return max_grades

_max_grade = build_max_grade(grades)
for year in grades.year.unique():
    for month in grades.month.unique():
        print(year, month, _max_grade[year][month])

2020 9 IV-
2020 11 V+
2020 12 6a
2020 4 None
2020 5 None
2020 7 None
2020 8 None
2020 10 None
2020 1 None
2020 2 None
2020 3 None
2020 6 None
2021 9 6b
2021 11 6b+
2021 12 6b+
2021 4 6b
2021 5 6b
2021 7 6a
2021 8 6a
2021 10 6b
2021 1 None
2021 2 None
2021 3 None
2021 6 None
2022 9 7a
2022 11 6b+
2022 12 7b
2022 4 7a
2022 5 7a
2022 7 7a
2022 8 6c
2022 10 6b
2022 1 6c+
2022 2 6c
2022 3 6c
2022 6 7a+
2023 9 6c
2023 11 7a
2023 12 7a+
2023 4 7a
2023 5 7a
2023 7 None
2023 8 6c+
2023 10 None
2023 1 7b
2023 2 6c
2023 3 7a
2023 6 None
2024 9 None
2024 11 None
2024 12 None
2024 4 6a
2024 5 6a
2024 7 None
2024 8 None
2024 10 None
2024 1 7a
2024 2 6b
2024 3 None
2024 6 None


#### Max OS grade

In [25]:
def build_max_os_grade(grades_df: pd.DataFrame):
    if "tries" not in grades_df:
        return None

    grades_os = grades_df[grades_df["tries"] == "OS"]

    max_grades = {}

    for yyyy in grades_os.year.unique():
        max_grades[int(yyyy)] = {}

        for mm in grades_os.month.unique():
            grades_month = grades_os[
                (grades_os.month == mm) &
                (grades_os.year == yyyy)
            ].grade

            try:
                max_grade_in_month = max(grades_month, key=lambda x: (x[0].isdigit(), x))
            except ValueError:
                max_grade_in_month = None
            max_grades[int(yyyy)][int(mm)] = max_grade_in_month
    return max_grades

_max_os_grade = build_max_os_grade(grades)
for year in grades.year.unique():
    for month in grades.month.unique():
        print(year, month, _max_os_grade[year][month])

2020 9 IV-
2020 11 None
2020 12 6a
2020 4 None
2020 5 None
2020 7 None
2020 8 None
2020 10 None
2020 1 None
2020 2 None
2020 3 None
2020 6 None
2021 9 6a+
2021 11 IV
2021 12 6a+
2021 4 6a
2021 5 IV+
2021 7 None
2021 8 6a
2021 10 6b
2021 1 None
2021 2 None
2021 3 None
2021 6 None
2022 9 6b
2022 11 V-
2022 12 6c
2022 4 6b
2022 5 6a+
2022 7 6c+
2022 8 6a+
2022 10 V
2022 1 6a+
2022 2 6c
2022 3 6a+
2022 6 6b+
2023 9 6b+
2023 11 None
2023 12 6b+
2023 4 6a+
2023 5 6a+
2023 7 None
2023 8 6a
2023 10 None
2023 1 6b+
2023 2 None
2023 3 None
2023 6 None
2024 9 None
2024 11 None
2024 12 None
2024 4 V+
2024 5 None
2024 7 None
2024 8 None
2024 10 None
2024 1 6c
2024 2 6a+
2024 3 None
2024 6 None


#### Total days

In [26]:
def build_total_days(dates_df: pd.DataFrame):
    total_days = {}

    for yyyy in dates_df.year.unique():
        total_days[int(yyyy)] = {}

        for mm in dates_df.month.unique():
            days = dates_df[
                (dates_df.month == mm) &
                (dates_df.year == yyyy)]
            total_days[int(yyyy)][int(mm)] = len(days)
    return total_days

_total_days = build_total_days(dates)
for year in grades.year.unique():
    for month in grades.month.unique():
        print(year, month, _total_days[year][month])

2020 9 4
2020 11 5
2020 12 6
2020 4 0
2020 5 0
2020 7 0
2020 8 1
2020 10 3
2020 1 0
2020 2 0
2020 3 0
2020 6 0
2021 9 4
2021 11 4
2021 12 6
2021 4 5
2021 5 7
2021 7 4
2021 8 1
2021 10 5
2021 1 1
2021 2 0
2021 3 2
2021 6 1
2022 9 5
2022 11 7
2022 12 8
2022 4 8
2022 5 11
2022 7 6
2022 8 7
2022 10 5
2022 1 5
2022 2 5
2022 3 4
2022 6 7
2023 9 9
2023 11 5
2023 12 8
2023 4 5
2023 5 8
2023 7 3
2023 8 7
2023 10 4
2023 1 5
2023 2 1
2023 3 6
2023 6 1
2024 9 0
2024 11 0
2024 12 0
2024 4 7
2024 5 6
2024 7 0
2024 8 0
2024 10 0
2024 1 3
2024 2 1
2024 3 2
2024 6 1


#### Total routes

In [27]:
def build_total_routes(grades_df: pd.DataFrame):
    total_routes = {}

    for yyyy in grades_df.year.unique():
        total_routes[int(yyyy)] = {}

        for mm in grades_df.month.unique():
            routes = grades_df[
                (grades_df.month == mm) &
                (grades_df.year == yyyy)]
            total_routes[int(yyyy)][int(mm)] = len(routes)
    return total_routes

_total_routes = build_total_routes(grades)
for year in grades.year.unique():
    for month in grades.month.unique():
        print(year, month, _total_routes[year][month])

2020 9 1
2020 11 1
2020 12 2
2020 4 0
2020 5 0
2020 7 0
2020 8 0
2020 10 0
2020 1 0
2020 2 0
2020 3 0
2020 6 0
2021 9 4
2021 11 4
2021 12 11
2021 4 2
2021 5 3
2021 7 3
2021 8 3
2021 10 7
2021 1 0
2021 2 0
2021 3 0
2021 6 0
2022 9 9
2022 11 14
2022 12 16
2022 4 10
2022 5 20
2022 7 17
2022 8 9
2022 10 4
2022 1 12
2022 2 11
2022 3 6
2022 6 13
2023 9 5
2023 11 2
2023 12 17
2023 4 5
2023 5 7
2023 7 0
2023 8 5
2023 10 0
2023 1 5
2023 2 3
2023 3 3
2023 6 0
2024 9 0
2024 11 0
2024 12 0
2024 4 4
2024 5 1
2024 7 0
2024 8 0
2024 10 0
2024 1 5
2024 2 3
2024 3 0
2024 6 0


#### Total pitches

In [28]:
def build_total_pitches(data: pd.DataFrame):
    if "pitches" not in data:
        return None

    total_pitches = {}

    for yyyy in data.year.unique():
        total_pitches[int(yyyy)] = {}

        for mm in data.month.unique():
            routes = data[
                (data.month == mm) &
                (data.year == yyyy)
            ]
            total_pitches[int(yyyy)][int(mm)] = sum(routes.pitches)

    return total_pitches

_total_pitches = build_total_pitches(trad_data)
for year in trad_data.year.unique():
    for month in trad_data.month.unique():
        print(year, month, _total_pitches[year][month])

2023 4 5
2023 5 7
2023 6 10
2023 7 3
2023 10 24
2023 12 5
2023 1 0
2024 4 0
2024 5 0
2024 6 0
2024 7 0
2024 10 0
2024 12 0
2024 1 6


#### Total climbed altitude

In [29]:
def build_total_distance(data: pd.DataFrame):
    if "length" not in data:
        return None

    total_length = {}

    for yyyy in data.year.unique():
        total_length[int(yyyy)] = {}

        for mm in data.month.unique():
            routes = data[
                (data.month == mm) &
                (data.year == yyyy)
            ]
            total_length[int(yyyy)][int(mm)] = sum(routes.length)

    return total_length

_total_length = build_total_distance(trad_data)
for year in trad_data.year.unique():
    for month in trad_data.month.unique():
        print(year, month, _total_length[year][month])

2023 4 170
2023 5 265
2023 6 320
2023 7 90
2023 10 720
2023 12 170
2023 1 0
2024 4 0
2024 5 0
2024 6 0
2024 7 0
2024 10 0
2024 12 0
2024 1 215


### Build json output

In [30]:
calendar_data = {
    "sport": CalendarData(sport),
    "boulder": CalendarData(boulder),
    "trad": CalendarData(trad),
    "multi": CalendarData(multi),
}

def generate_json(data: pd.DataFrame):
    return {
        "days": build_calendar(data.dates),
        "grades": build_grades_hist(data.grades),
        "routes": build_routes_hist(data.grades),
        "pitches": build_pitches_calendar(data.grades),
        "distance": build_altitude_calendar(data.grades),
        "labels": {
            "max_grade": build_max_grade(data.grades),
            "max_os_grade": build_max_os_grade(data.grades),
            "total_days": build_total_days(data.dates),
            "total_routes": build_total_routes(data.grades),
            "total_pitches": build_total_pitches(data.grades),
            "total_distance": build_total_distance(data.grades),
        }
    }

calendar_json = {
    "sport": generate_json(calendar_data["sport"]),
    "boulder": generate_json(calendar_data["boulder"]),
    "trad": generate_json(calendar_data["trad"]),
    "multi": generate_json(calendar_data["multi"]),
}

with open("mock_calendar.json", "w", encoding="utf-8") as f:
    f.write(json.dumps(calendar_json))

C:\Users\Alonso\AppData\Local\Temp\ipykernel_14200\1183014363.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  grades["year"] = grades["date"].dt.year
C:\Users\Alonso\AppData\Local\Temp\ipykernel_14200\1183014363.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  grades["month"] = grades["date"].dt.month
C:\Users\Alonso\AppData\Local\Temp\ipykernel_14200\1183014363.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer]